In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatOpenAI(
    model="prism-ml/ternary-bonsai-2-27b",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=1000,
)

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print(os.getcwd())

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8396\635496276.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


c:\Users\Lenovo\Desktop\Agentic AI\GuardRail_RAG_project


In [13]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

### Read all the pdf's inside the directory
data_dir = Path("data")
pdf_files = sorted(data_dir.glob("**/*.pdf"))

print(f"Found {len(pdf_files)} PDF files")

documents = []

for pdf_file in pdf_files:
    try:
        loader = PyPDFLoader(str(pdf_file))
        pages = loader.load()

        for page in pages:
            page.metadata["source"] = pdf_file.name

        documents.extend(pages)
        print(f"Loaded: {pdf_file.name} ({len(pages)} pages)")

    except Exception as e:
        print(f"Failed: {pdf_file.name} | {e}")

print(f"\nTotal pages loaded: {len(documents)}")

Found 14 PDF files
Loaded: Accounting_and_Disclosure-Policy-2025.pdf (36 pages)
Loaded: Anti-Harrsasement-Policy.pdf (8 pages)
Loaded: Banking-Companies-Ordinance-1962.pdf (202 pages)
Loaded: Banking-on-Equality.pdf (6 pages)
Loaded: BankIslami-Fair-Treatment-of-Consumers-Policy-Framework-SOP.pdf (12 pages)
Loaded: Banks_CFI_AML.pdf (29 pages)
Loaded: C9-Enclosure-2.pdf (37 pages)
Loaded: Code-of-conduct-2021.pdf (13 pages)
Loaded: Enterprise-Risk-Management-Framework.pdf (28 pages)
Loaded: Global-Whistle-Blow-Speak-Up-Policy-v4.pdf (11 pages)
Loaded: Investor-Relations-Policy.pdf (5 pages)
Loaded: MarketingCommunications_PR Policy-2025.pdf (11 pages)
Loaded: Policy-for-Financial-Inclusion-of-PWDs.pdf (26 pages)
Loaded: SBP-Guidelines-of-Business-Conduct-for-Banks-.pdf (27 pages)

Total pages loaded: 451


In [14]:
for doc in documents[:3]:
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print("Text preview:", repr(doc.page_content[:400]))
    print("-" * 50)

Source: Accounting_and_Disclosure-Policy-2025.pdf
Page: 0
Text preview: 'ACCOUNTING & DISCLOSURE POLICY FINANCIAL CONTROL GROUP \n \n \nACCOUNTING & DISCLOSURE POLICY \n \nFINANCIAL CONTROL GROUP'
--------------------------------------------------
Source: Accounting_and_Disclosure-Policy-2025.pdf
Page: 1
Text preview: 'ACCOUNTING & DISCLOSURE POLICY FINANCIAL CONTROL GROUP \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nThis document is for internal use of National Bank of Pakistan employees only. \nAny act of divulgence to third parties shall be viewed seriously and warrant \ndisciplinary action.'
--------------------------------------------------
Source: Accounting_and_Disclosure-Policy-2025.pdf
Page: 2
Text preview: 'ACCOUNTING & DISCLOSURE POLICY FINANCIAL CONTROL GROUP \n \n \n \nTable of Contents \n1. PREAMBLE .................................................................................... 2 \n2. ACCOUNTING & FINANCIAL REPORTING FRAMEWORK .. 2 \n3. MATE

In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

if chunks:
    print("\nExample chunk:")
    print(chunks[0].page_content[:500])
    print("\nMetadata:", chunks[0].metadata)

Total chunks: 1278

Example chunk:
ACCOUNTING & DISCLOSURE POLICY FINANCIAL CONTROL GROUP 
 
 
ACCOUNTING & DISCLOSURE POLICY 
 
FINANCIAL CONTROL GROUP

Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-12-19T17:27:04+05:00', 'author': 'Ehtisham Ahmad', 'keywords': 'ADP', 'moddate': '2025-12-19T17:27:04+05:00', 'source': 'Accounting_and_Disclosure-Policy-2025.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'start_index': 0}


In [ ]:
# First time creation of FAISS vector store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create FAISS vector store from chunks
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)


vectorstore.save_local("faiss_vector_store")

print("FAISS vector store created and saved successfully!")

In [5]:
#if store already exists then run this cell nigga
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.load_local(
    "faiss_vector_store",
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS vector store loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13352.29it/s]


FAISS vector store loaded successfully!


In [6]:
# testing query retrieval 
query = "What is Lease liability against right of use assets?"

retrieved_docs = vectorstore.similarity_search(
    query,
    k=4
)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:700])


--- Result 1 ---
Source: Accounting_and_Disclosure-Policy-2025.pdf
Page: 15
rate that the Bank would have to pay to borrow the funds necessary to obtain an asset of 
similar value in a similar economic environment with similar terms and conditions. 
 
On initial recognition, right-of-use assets are measured at cost comprising the following: 
 
• the amount of the initial measurement of lease liability; 
• any lease payments made at or before the commencement date less any 
lease incentives received; 
• any initial direct costs incurred; and 
• an estimate of restoration costs. 
 
The Bank leases various offices / branches for the purpose of its operational activities. 
Rental contracts are typically made for fixed periods of 3 to 10 years. Lease terms are 
negotiate

--- Result 2 ---
Source: Accounting_and_Disclosure-Policy-2025.pdf
Page: 10
ACCOUNTING & DISCLOSURE POLICY FINANCIAL CONTROL GROUP 
 
 
8 |P a g e  
 
 
h) Provision against contingencies 
 
Provision against contingencie

In [ ]:
#-----------------------------

In [7]:
def rag_answer(query: str) -> str:
    """Answer from indexed PDFs, or return a sentinel if the PDFs cannot answer."""
    retrieved_docs = vectorstore.similarity_search(query, k=4)

    if not retrieved_docs:
        return "NO_RELEVANT_DOCUMENTS_FOUND"

    context_parts = []
    references = []
    seen = set()

    for doc in retrieved_docs:
        source = doc.metadata.get("source", "Unknown")
        page_index = doc.metadata.get("page")

        page_number = page_index + 1 if isinstance(page_index, int) else page_index

        context_parts.append(
            f"Source: {source}\nPage: {page_number}\nContent: {doc.page_content}"
        )

        ref = (source, page_number)
        if ref not in seen:
            seen.add(ref)
            references.append(f"- {source}, page {page_number}")

    context = "\n\n".join(context_parts)
    prompt = f"""
You answer questions using ONLY the supplied PDF context.
If the context does not contain enough information to answer the question,
return exactly this token and nothing else: NO_RELEVANT_DOCUMENTS_FOUND
Do not guess or use outside knowledge.
Do not write a Sources/References section; the application adds references.

PDF context:
{context}

Question: {query}

Answer:
"""

    response = llm.invoke(prompt)
    answer = response.content.strip()

    if "NO_RELEVANT_DOCUMENTS_FOUND" in answer:
        return "NO_RELEVANT_DOCUMENTS_FOUND"

    if not answer:
        return "NO_RELEVANT_DOCUMENTS_FOUND"

    return f"{answer}\n\nSources:\n" + "\n".join(references)


In [8]:
query = "What is Lease liability against right of use assets?"

answer = rag_answer(query)

print(answer)

Lease liability against right-of-use assets refers to the Bank’s lease accounting as lessee. The lease liability is initially measured at the present value of lease payments, discounted using the incremental borrowing rate. The related right-of-use asset is measured at cost, including the initial lease liability, initial lease payments less incentives, initial direct costs, and estimated restoration costs.

The Bank applies judgement in determining the lease term, including whether renewal options are reasonably certain to be exercised, and in establishing the discount rate for new leases, using the incremental borrowing rate as a benchmark.

Sources:
- Accounting_and_Disclosure-Policy-2025.pdf, page 16
- Accounting_and_Disclosure-Policy-2025.pdf, page 11
- Accounting_and_Disclosure-Policy-2025.pdf, page 19


In [ ]:
=

In [4]:
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a banking guide with expertise in banking policies, procedures and services.
Always provide concrete examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I withdraw money from my account?")
]
response = llm.invoke(messages)
print(response.content)



To withdraw money from your account, use one of these common methods:

1. **ATM withdrawal**
   - Go to an ATM.
   - Insert your debit card and enter your PIN.
   - Select “Withdraw” and choose an amount.
   - **Example:** At a bank ATM, you withdraw $100 in cash.

2. **Bank branch/teller withdrawal**
   - Visit a branch with your government-issued ID.
   - Ask a teller to withdraw cash.
   - **Example:** You bring your driver’s license and debit card and request a $200 cash withdrawal.

3. **Mobile app or online banking transfer**
   - Log in to your bank’s app or website.
   - Choose “Transfers” or “Withdraw.”
   - Select the source and destination accounts.
   - **Example:** You transfer $500 from your checking account to your savings account.

4. **Check or money order**
   - Ask your bank to write a check to yourself or pay a bill.
   - Some banks require you to be present or call ahead.
   - **Example:** You request a $300 check, which you can deposit or cash later.

Important 